# Etherpad Pad 内容重建查看器

本工具用于：
1. 查询数据库获取所有 room-id 和版本范围
2. SSH 连接到服务器执行重建脚本
3. 获取并展示重建的内容
4. 支持编辑和对比

---

## 📋 功能清单

- ✅ 查询 MySQL 数据库获取所有 Pad 列表
- ✅ 显示每个 Pad 的版本范围
- ✅ SSH 连接到远程服务器执行重建脚本
- ✅ 查看特定版本的详细内容


## 使用说明

### 完整使用流程

1. **安装依赖** - 运行第1个单元格
2. **导入配置** - 运行第2个单元格
3. **定义函数** - 运行第3-4个单元格
4. **查询 Pad 列表** - 运行第5个单元格,查看所有可用的 Pad
5. **执行重建** - 修改第6个单元格的参数,然后运行
6. **查看结果** - 运行后续单元格查看详细内容

### 参数说明

```python
# 第6个单元格 - 执行重建
pad_id = 'room-229'    # 从查询结果中选择 Pad ID
start_rev = 0          # 起始版本号（通常从0开始）
end_rev = 10           # 结束版本号

# 第9个单元格 - 查看特定版本
view_revision = 5      # 要查看的版本号

# 第10个单元格 - 版本对比
compare_rev1 = 5       # 对比版本1
compare_rev2 = 6       # 对比版本2
```

### 示例工作流

```python
# 1. 找到你感兴趣的 Pad
# 运行第5个单元格后，从输出表格中找到 pad_id

# 2. 重建内容
pad_id = 'room-229'
start_rev = 0
end_rev = 50
# 运行第6个单元格

# 3. 查看特定版本
view_revision = 10
# 运行第9个单元格

# 4. 对比版本
compare_rev1 = 10
compare_rev2 = 20
# 运行第10个单元格

# 5. 导出结果
# 运行第11个单元格
```

## 1. 安装依赖包

首次使用需要安装必要的 Python 包


In [ ]:
# 安装必要的 Python 包
#!pip install pymysql paramiko pandas ipywidgets -q


## 2. 导入库和配置


In [7]:
import pymysql
import paramiko
import json
import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 数据库配置
DB_CONFIG = {
    'host': '112.74.92.135',
    'user': 'root',
    'password': '1q2w3e4R',
    'database': 'alic',
    'charset': 'utf8mb4',
    'port': 3306
}

# SSH 配置
SSH_CONFIG = {
    'host': '8.138.89.124',
    'port': 22,
    'username': 'root',
    'password': 'Alic2025'
}

# 脚本路径
SCRIPT_PATH = '/opt/etherpad-test/src/node/scheduler/etherpad_changes/PadContentRebuildAPI.js'
WORK_DIR = '/opt/etherpad-test/src'

print("✅ 配置加载完成")


✅ 配置加载完成


## 3. 定义数据库查询函数


In [8]:
def get_all_room_ids_with_versions():
    """从 store 表查询所有 room-id 及其版本范围"""
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        query = """
        SELECT 
            SUBSTRING_INDEX(SUBSTRING_INDEX(`key`, ':', 2), ':', -1) as pad_id,
            MIN(CAST(SUBSTRING_INDEX(`key`, ':', -1) AS UNSIGNED)) as min_revision,
            MAX(CAST(SUBSTRING_INDEX(`key`, ':', -1) AS UNSIGNED)) as max_revision,
            COUNT(*) as revision_count
        FROM store
        WHERE `key` LIKE 'pad:%:revs:%'
        GROUP BY pad_id
        ORDER BY pad_id
        """
        
        cursor.execute(query)
        results = cursor.fetchall()
        
        cursor.close()
        connection.close()
        
        return results
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}")
        return []

print("✅ 数据库查询函数定义完成")


✅ 数据库查询函数定义完成


## 4. 定义 SSH 远程执行函数


In [ ]:
import base64

def decode_base64_field(value):
    """解码 Base64 字段"""
    if not value:
        return ''
    try:
        return base64.b64decode(value).decode('utf-8')
    except Exception:
        return value  # 如果解码失败，返回原值

def execute_rebuild_script(pad_id, start_rev=None, end_rev=None, use_base64=False):
    """
    SSH 连接到服务器执行重建脚本
    
    参数:
        pad_id: Pad ID
        start_rev: 起始版本号
        end_rev: 结束版本号
        use_base64: 是否使用 Base64 编码模式（推荐用于大数据量）
    """
    try:
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        
        print(f"🔗 连接到服务器 {SSH_CONFIG['host']}...")
        ssh.connect(
            hostname=SSH_CONFIG['host'],
            port=SSH_CONFIG['port'],
            username=SSH_CONFIG['username'],
            password=SSH_CONFIG['password'],
            timeout=30
        )
        print("✅ SSH 连接成功")
        
        # 构建命令
        cmd_parts = [
            f"cd {WORK_DIR}",
            "&&",
            "node --require tsx/cjs",
            SCRIPT_PATH,
            pad_id
        ]
        
        if start_rev is not None:
            cmd_parts.append(str(start_rev))
            if end_rev is not None:
                cmd_parts.append(str(end_rev))
        
        # 如果使用 Base64 模式，添加参数
        if use_base64:
            cmd_parts.append('--base64')
            print("📦 使用 Base64 编码模式")
        else:
            print("📝 使用标准模式")
        
        command = ' '.join(cmd_parts)
        print(f"🚀 执行命令: {command}")
        
        # 执行命令，增加超时时间
        stdin, stdout, stderr = ssh.exec_command(command, timeout=600)
        
        # 读取输出
        output = stdout.read().decode('utf-8')
        error = stderr.read().decode('utf-8')
        exit_code = stdout.channel.recv_exit_status()
        
        ssh.close()
        
        if exit_code == 0:
            print("✅ 脚本执行成功")
            try:
                result = json.loads(output)
                
                # 如果是 Base64 编码模式，解码数据
                if result.get('encoding') == 'base64':
                    print("🔓 解码 Base64 数据...")
                    for version in result.get('versions', []):
                        if 'content_base64' in version:
                            version['content'] = decode_base64_field(version['content_base64'])
                            del version['content_base64']
                        if 'changeset_base64' in version:
                            version['changeset'] = decode_base64_field(version['changeset_base64'])
                            del version['changeset_base64']
                        if 'attribs_base64' in version:
                            version['attribs'] = decode_base64_field(version['attribs_base64'])
                            del version['attribs_base64']
                    print("✅ Base64 解码完成")
                
                return result
            except json.JSONDecodeError as e:
                print(f"❌ JSON 解析失败: {e}")
                print(f"输出前500字符: {output[:500]}")
                print(f"输出后500字符: {output[-500:]}")
                return {'success': False, 'error': 'JSON parse error', 'details': str(e)}
        else:
            print(f"❌ 脚本执行失败 (退出码: {exit_code})")
            print(f"错误信息: {error}")
            return {'success': False, 'error': error, 'exit_code': exit_code}
            
    except Exception as e:
        print(f"❌ SSH 执行失败: {e}")
        import traceback
        traceback.print_exc()
        return {'success': False, 'error': str(e)}

print("✅ SSH 执行函数定义完成（支持标准模式和 Base64 模式）")


✅ SSH 执行函数定义完成（支持标准模式和 Base64 模式）


## 5. 查询所有 Room ID 和版本范围


In [10]:
# 查询所有 room-id 及版本范围
print("🔍 正在查询数据库...")
room_list = get_all_room_ids_with_versions()

if room_list:
    df = pd.DataFrame(room_list)
    df.index = df.index + 1
    
    print(f"\n✅ 找到 {len(room_list)} 个 Pad\n")
    display(df)
    
    # 统计信息
    total_revisions = df['revision_count'].sum()
    avg_revisions = df['revision_count'].mean()
    
    print(f"\n📊 统计信息:")
    print(f"   总 Pad 数: {len(room_list)}")
    print(f"   总版本数: {total_revisions}")
    print(f"   平均版本数: {avg_revisions:.2f}")
    print(f"   最多版本: {df['revision_count'].max()} (Pad: {df.loc[df['revision_count'].idxmax(), 'pad_id']})")
    print(f"   最少版本: {df['revision_count'].min()} (Pad: {df.loc[df['revision_count'].idxmin(), 'pad_id']})")
else:
    print("❌ 未找到任何 Pad 数据")
    df = pd.DataFrame()


🔍 正在查询数据库...

✅ 找到 9 个 Pad



,pad_id,min_revision,max_revision,revision_count
1,room-165,0,0,1
2,room-210,0,0,1
3,room-229,0,159,160
4,room-243,0,26,27
5,room-254,0,0,1
6,room-255,0,0,1
7,room-258,0,8,9
8,room-260,0,202,203
9,room-261,0,13,14



📊 统计信息:
   总 Pad 数: 9
   总版本数: 417
   平均版本数: 46.33
   最多版本: 203 (Pad: room-260)
   最少版本: 1 (Pad: room-165)


## 6. 执行 Pad 内容重建

💡 **使用说明：** 修改下面代码中的参数，然后运行单元格

- `pad_id`: 从上面表格中选择一个 Pad ID
- `start_rev`: 起始版本号
- `end_rev`: 结束版本号


In [ ]:
# ========================================
# 手动设置参数并执行重建
# ========================================

# 📝 在这里修改参数
pad_id = 'room-260'        # 修改为你要查询的 Pad ID
start_rev = 0              # 起始版本号
end_rev = 202              # 结束版本号
use_base64 = True          # 使用 Base64 编码模式（推荐用于大数据量和远程调用）

# 执行重建
if not df.empty:
    print(f"\n{'='*80}")
    print(f"📝 Pad ID: {pad_id}")
    print(f"📊 版本范围: {start_rev} - {end_rev}")
    print(f"🔧 编码模式: {'Base64（推荐）' if use_base64 else '标准模式'}")
    print(f"⏰ 开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    
    # 执行重建
    rebuild_result = execute_rebuild_script(pad_id, start_rev, end_rev, use_base64=use_base64)
    
    if rebuild_result.get('success'):
        stats = rebuild_result.get('statistics', {})
        print(f"\n{'='*80}")
        print(f"✅ 重建完成！")
        print(f"{'='*80}")
        print(f"   总版本数: {stats.get('total', 0)}")
        print(f"   成功: {stats.get('success', 0)}")
        print(f"   失败: {stats.get('failed', 0)}")
        print(f"\n⏰ 完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"\n💡 结果已保存到变量 rebuild_result，可以在下面的单元格中查看详细内容")
    else:
        print(f"\n{'='*80}")
        print(f"❌ 重建失败")
        print(f"{'='*80}")
        print(f"错误信息: {rebuild_result.get('error', 'Unknown error')}")
else:
    print("❌ 没有可用的 Pad 数据，请先执行上面的查询")



📝 Pad ID: room-260
📊 版本范围: 0 - 202
🔧 编码模式: Base64（推荐）
⏰ 开始时间: 2025-12-24 13:25:54

🔗 连接到服务器 8.138.89.124...
✅ SSH 连接成功
📦 使用 Base64 编码模式
🚀 执行命令: cd /opt/etherpad-test/src && node --require tsx/cjs /opt/etherpad-test/src/node/scheduler/etherpad_changes/PadContentRebuildAPI.js room-260 0 202 --base64
✅ 脚本执行成功
❌ JSON 解析失败: Unterminated string starting at: line 1055 column 18 (char 63453)
输出前500字符: {
  "success": true,
  "pad_id": "room-260",
  "head_revision": 202,
  "requested_range": {
    "start": 0,
    "end": 202
  },
  "statistics": {
    "total": 203,
    "success": 203,
    "failed": 0
  },
  "versions": [
    {
      "revision": 0,
      "success": true,
      "pad_id": "room-260",
      "content": "Welcome to Etherpad!\n\nThis pad text is synchronized as you type, so that everyone viewing this page sees the same text. This allows you to collaborate seamlessly on documents!\n\nGe
输出后500字符: angeset": "Z:dl>2=b*1+2$文字",
      "attribs": "*1+e|1+1*0|1+1*0+6*1|6+6m*1+6e|1+1",
      "

In [ ]:
## 7. 版本详情查看器 - 查看对应版本的内容

In [ ]:
# 取出版本列表
versions = rebuild_result["versions"]

# 转 DataFrame
revision_content = pd.DataFrame([
    {"revision": v["revision"], "content": v["content"]}
    for v in versions
])

revision_content

,revision,content
0,0,Welcome to Etherpad!\n\nThis pad text is synch...
1,1,Welcome to Etherpad!\n\n
2,2,Welcome to Etherpad!\n\n周末
3,3,Welcome to Etherpad!\n\n周末去哪里玩
4,4,Welcome to Etherpad!\n\n周末去哪里玩\n
5,5,Welcome to Etherpad!\n\n周末去哪里玩\n可以去迪士尼二元
6,6,Welcome to Etherpad!\n\n周末去哪里玩\n可以去迪士尼
7,7,Welcome to Etherpad!\n\n周末去哪里玩\n可以去迪士尼乐园
8,8,Welcome to Etherpad!\n\n周末去哪里玩\n可以去迪士尼乐园1
9,9,Welcome to Etherpad!\n\n周末去哪里玩\n可以去迪士尼乐园12312312


## 8. 查看具体版本的详细内容

修改 `view_revision` 参数来查看不同版本


In [ ]:
# 📝 修改这里的版本号
view_revision = 5

# 查看版本详情
if 'rebuild_result' in globals() and rebuild_result and rebuild_result.get('success'):
    versions = rebuild_result.get('versions', [])
    version = next((v for v in versions if v.get('revision') == view_revision), None)
    
    if version and version.get('success'):
        print(f"\n{'='*80}")
        print(f"📝 版本 {view_revision} 详细信息")
        print(f"{'='*80}")
        print(f"Pad ID:        {version.get('pad_id', 'N/A')}")
        print(f"作者:          {version.get('author', 'N/A')}")
        print(f"时间戳:        {version.get('timestamp', 'N/A')}")
        print(f"格式化时间:    {version.get('formatted_timestamp', 'N/A')}")
        print(f"文本长度:      {version.get('text_length', 0)} 字符")
        print(f"行数:          {version.get('line_count', 0)}")
        print(f"变更摘要:      {version.get('change_summary', 'N/A')}")
        
        print(f"\n{'='*80}")
        print(f"📄 文本内容")
        print(f"{'='*80}")
        print(version.get('content', ''))
        
        print(f"\n{'='*80}")
        print(f"🔧 Changeset")
        print(f"{'='*80}")
        print(version.get('changeset', 'N/A'))
        
        print(f"\n{'='*80}")
        print(f"🎨 Attributes")
        print(f"{'='*80}")
        attribs = version.get('attribs', '')
        print(attribs if attribs else '(无属性)')
    else:
        print(f"❌ 未找到版本 {view_revision}")
else:
    print("⚠️ 请先执行重建")



📝 版本 5 详细信息
Pad ID:        room-229
作者:          a.ni6xvsCFoJs9Rr1v
时间戳:        1758377771664
格式化时间:    2025-09-20 22:16:11.664
文本长度:      37 字符
行数:          4
变更摘要:      30 -> 38 chars

📄 文本内容
Welcome to Etherpad!

周末去哪里玩
可以去迪士尼二元

🔧 Changeset
Z:u>8|3=t*0+8$可以去迪士尼二元

🎨 Attributes
|2+m*0|1+7*0+8|1+1


## 9. 读取 pad_version_changes 表数据

从数据库中读取指定 pad 的版本变化记录

In [ ]:
def get_pad_version_changes(room_id=None):
    """从数据库中读取指定 pad 的版本变化记录"""
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        if room_id:
            query = """SELECT * FROM pad_version_changes WHERE pad_id = %s ORDER BY seq_order ASC"""
            cursor.execute(query, [room_id])
        else:
            query = """SELECT * FROM pad_version_changes ORDER BY pad_id, seq_order ASC"""
            cursor.execute(query)
        
        results = cursor.fetchall()
        
        cursor.close()
        connection.close()
        
        return results
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}")
        return []

print("✅ 数据库查询函数定义完成")

✅ 数据库查询函数定义完成


In [ ]:
# 读取数据
room_id = "room-229"
df_versions = get_pad_version_changes(room_id=room_id)
df_versions = pd.DataFrame(df_versions)
if df_versions is not None and len(df_versions) > 0:
    df = df_versions[[
    "pad_id",
    "seq_order",
    "behavior",
    "author",
    "content",
    "add_start_time",
    "add_end_time",
    "delete_start_time",
    "delete_end_time"
]]

df_versions.head(5)

,pad_id,seq_order,behavior,author,content,add_start_time,add_end_time,delete_start_time,delete_end_time
0,room-229,1,deleted,a.rVEwX679hNTTNivd,*,2025-09-26 23:05:32.012,2025-09-26 23:05:32.012,2025-10-21 22:47:16.498,2025-10-21 22:47:16.498
1,room-229,2,add,a.rVEwX679hNTTNivd,欢迎来到,2025-10-21 22:47:34.952,2025-10-21 22:47:34.952,None,None
2,room-229,3,add,a.ni6xvsCFoJs9Rr1v,Welcome to Etherpad!\n\n,2025-09-17 23:39:12.431,2025-09-17 23:39:12.431,None,None
3,room-229,4,deleted,a.ni6xvsCFoJs9Rr1v,"This pad text is synchronized as you type, so ...",2025-09-17 23:39:12.431,2025-09-17 23:39:12.431,2025-09-20 22:15:52.139,2025-09-20 22:15:52.139
4,room-229,5,add,a.ni6xvsCFoJs9Rr1v,周末去哪里玩\n可以去迪士尼,2025-09-20 22:15:58.322,2025-09-20 22:16:11.664,None,None
